## Toon vs Json in Information Extraction

In [ ]:
# ! pip install -q openai datasets pandas tqdm dotenv mlflow

In [ ]:
# ! pip install git+https://github.com/toon-format/toon-python.git

### Imports

In [1]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import pandas as pd
import mlflow
from utils import (
    generate_urls,
    calculate_individual_invoice_accuracies,
)

from dotenv import load_dotenv
from mlflow.entities import Feedback
from mlflow.genai import scorer
from toon_format import encode, decode


load_dotenv()

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


True

### Config

In [2]:
MLFLOW_TRACKING_URI = "http://localhost:5000/"
MODEL_NAME = "gpt-5-mini"
REASONING = "medium"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gpt5-mini"
FORMAT = "toon"

### Initialize MLflow and OpenAI environment

In [ ]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
mlflow.openai.autolog()

### Load the dataset

In [3]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

In [ ]:
json_example = json.loads(dataset['test'][0]['ground_truth'])['gt_parse']
toon_example = encode(json_example)
print(toon_example)

### Register prompt and model

In [ ]:
PROMPT_NAME = f"invoice-extraction-{FORMAT}-prompt"
PROMPT_VERSION = "1"
if FORMAT == "toon":
    system_prompt = """You are a Vision Language Model designed to extract structured data from invoice receipts.
        Task:
        Convert the invoice receipt into a TOON format - strictly following the schema provided.

        Requirements:
        1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
        2. Preserve exact formatting for all the extracted values.  
        3. Do not output fields that lack data—omit empty keys.  
        4. Do not add any information not present in the invoice.
        5. In case of prices and currencies, ensure to maintain the original format without any modifications.

        Schema:
        ```toon
        {{schema}}
        ```

        Output:
        Return the output in TOON format matching this schema - no extraneous keys or null values.

        For example:
        ```toon
        {{example}}
        ```
        """
else:
    system_prompt = """You are a Vision Language Model designed to extract structured data from invoice receipts.
        Task:
        Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

        Requirements:
        1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
        2. Preserve exact formatting for all the extracted values.  
        3. Do not output fields that lack data—omit empty keys.  
        4. Do not add any information not present in the invoice.
        5. In case of prices and currencies, ensure to maintain the original format without any modifications.

        Schema:
        {{schema}}

        Output:
        Return valid, minimal JSON matching this schema - no extraneous keys or null values.

        For example:
        ```json
        {{example}}
        ```
        """
mlflow.genai.register_prompt(name=PROMPT_NAME, template=system_prompt)


### Data Preparation

In [ ]:
if FORMAT == "toon":
    with open("schema.toon", "r") as f:
        schema_dict = f.read()
else:
    with open("schema.json", "r") as f:
        schema_dict = json.load(f)

print(schema_dict)

In [ ]:
NUM_SAMPLES = 10
test_dataset = dataset["test"].select(range(1, NUM_SAMPLES + 1))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

### Inference and Evaluation

In [ ]:
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")

eval_dataset = []
for index, url in enumerate(url_list):
    eval_dict = {
        "inputs": {"image_base64": url, "schema": schema_dict},
        "expectations": {"expected_response": ground_truth_list[index]},
    }
    eval_dataset.append(eval_dict)

In [ ]:
def predict_fn(image_base64, schema) -> str:
    system_prompt_template = mlflow.genai.load_prompt(
        name_or_uri=PROMPT_NAME, version=PROMPT_VERSION
    )

    if FORMAT == "toon":
        system_prompt = system_prompt_template.format(schema=schema, example=toon_example)
    else:
        system_prompt = system_prompt_template.format(schema=schema, example=json_example)

    response = client.responses.create(
        model=MODEL_NAME,
        reasoning={
            "effort": REASONING,
        },
        text={"verbosity": "low"},
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": system_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image_base64}",
                    },
                ],
            }
        ],
    )
    response_text = response.output[1].content[0].text
    # print(response_text)

    return response_text

In [ ]:
@scorer
def exact_match(inputs, outputs, expectations, trace) -> Feedback:
    if FORMAT == "toon":
        try:
            outputs = decode(outputs)
        except:
            print("Unable to decode the output")
            return 0
    else:
        outputs = json.loads(outputs)
    expectations = expectations["expected_response"]
    _, acc = calculate_individual_invoice_accuracies(
        ground_truth=expectations, output=outputs
    )
    acc = round(acc, 2)
    print(f"Accuracy: {acc}")
    return acc


In [ ]:
with mlflow.start_run(run_name=f"eval-cord-{FORMAT}") as run:
    results = mlflow.genai.evaluate(
        data=eval_dataset,
        scorers=[exact_match],
        predict_fn=predict_fn,
    )

In [ ]:
def retrieve_token_usage(trace_df):
    input_tokens_list = []
    output_tokens_list = []
    reasoning_tokens_list = []
    for i in range(len(trace_df)):
        token_dict = trace_df["response"][i]["usage"]
        input_tokens = token_dict.get("input_tokens", 0)
        reasoning_tokens = token_dict.get("output_tokens_details", 0).get(
            "reasoning_tokens", 0
        )
        output_tokens = token_dict.get("output_tokens", 0)
        input_tokens_list.append(input_tokens)
        output_tokens_list.append(output_tokens)
        reasoning_tokens_list.append(reasoning_tokens)

    return input_tokens_list, output_tokens_list, reasoning_tokens_list

In [ ]:
trace_df = mlflow.search_traces(run_id=run.info.run_id)
i_tokens, o_tokens, r_tokens = retrieve_token_usage(trace_df)

In [ ]:
def get_accuracy_list(trace_df):
    accuracy_list = []
    for assesment_list in trace_df['assessments']:
        for assesment in assesment_list:
            if assesment['assessment_name'] == "exact_match":
                accuracy_list.append(assesment['feedback']['value'])
    return accuracy_list

accuracy_list = get_accuracy_list(trace_df)
accuracy_list

In [ ]:
total_tokens = sum(i_tokens) + sum(o_tokens) + sum(r_tokens)
total_tokens

In [ ]:
df = pd.DataFrame(
    {
        f"{FORMAT}_input_tokens": i_tokens,
        f"{FORMAT}_output_tokens": o_tokens,
        f"{FORMAT}_reasoning_tokens": r_tokens,
        f"{FORMAT}_accuracy": accuracy_list
    }
)

df.to_csv(f"{FORMAT}_token_usage.csv", index=False)

In [ ]:
toon_df = pd.read_csv("toon_token_usage.csv")
json_df = pd.read_csv("json_token_usage.csv")

In [ ]:
merged_df = pd.concat([toon_df, json_df], axis=1)
merged_df

In [ ]:
# Calculate the sums
sum_df = merged_df[merged_df.columns].sum().reset_index(name="sum")

In [ ]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare sums for bar chart
bar_labels = [
    "Toon Input", "JSON Input",
    "Toon Output", "JSON Output",
    "Toon Reasoning", "JSON Reasoning"
]
bar_keys = [
    "toon_input_tokens", "json_input_tokens",
    "toon_output_tokens", "json_output_tokens",
    "toon_reasoning_tokens", "json_reasoning_tokens"
]
bar_values = []
for key in bar_keys:
    value = sum_df[sum_df['index'] == key]['sum'].values[0]
    bar_values.append(value)
bar_colors = ['royalblue', 'seagreen', 'royalblue', 'seagreen', 'royalblue', 'seagreen']

# Create 2x2 subplot grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Input Tokens Comparison",
        "Output Tokens Comparison",
        "Reasoning Tokens Comparison",
        "Toon vs JSON Token Sums"
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

# Plot 1: Input Tokens
fig.add_trace(
    go.Bar(
        name='Toon Input Tokens',
        x=merged_df.index,
        y=merged_df['toon_input_tokens'],
        marker_color='royalblue'
    ),
    row=1, col=1
)
fig.add_trace(
    go.Bar(
        name='JSON Input Tokens',
        x=merged_df.index,
        y=merged_df['json_input_tokens'],
        marker_color='seagreen'
    ),
    row=1, col=1
)

# Plot 2: Output Tokens
fig.add_trace(
    go.Bar(
        name='Toon Output Tokens',
        x=merged_df.index,
        y=merged_df['toon_output_tokens'],
        marker_color='royalblue'
    ),
    row=1, col=2
)
fig.add_trace(
    go.Bar(
        name='JSON Output Tokens',
        x=merged_df.index,
        y=merged_df['json_output_tokens'],
        marker_color='seagreen'
    ),
    row=1, col=2
)

# Plot 3: Reasoning Tokens
fig.add_trace(
    go.Bar(
        name='Toon Reasoning Tokens',
        x=merged_df.index,
        y=merged_df['toon_reasoning_tokens'],
        marker_color='royalblue'
    ),
    row=2, col=1
)
fig.add_trace(
    go.Bar(
        name='JSON Reasoning Tokens',
        x=merged_df.index,
        y=merged_df['json_reasoning_tokens'],
        marker_color='seagreen'
    ),
    row=2, col=1
)

# Plot 4: Token Sums
fig.add_trace(
    go.Bar(
        x=bar_labels,
        y=bar_values,
        marker_color=bar_colors,
        showlegend=False
    ),
    row=2, col=2
)

# Update x and y axis titles for each subplot
fig.update_xaxes(title_text="Invoices", row=1, col=1)
fig.update_yaxes(title_text="Input Tokens", row=1, col=1)

fig.update_xaxes(title_text="Invoices", row=1, col=2)
fig.update_yaxes(title_text="Output Tokens", row=1, col=2)

fig.update_xaxes(title_text="Invoices", row=2, col=1)
fig.update_yaxes(title_text="Reasoning Tokens", row=2, col=1)

fig.update_xaxes(title_text="Token Type", row=2, col=2)
fig.update_yaxes(title_text="Total Tokens", row=2, col=2)

# Set grouped bar mode and layout
fig.update_layout(
    barmode='group',
    title_text="Toon vs JSON Token Usage Comparison",
    height=800,
    width=950
)

fig.show()


### Raw data comparison

In [17]:
from tqdm import tqdm

results = {}
splits = ["train", "test", "validation"]
for split in splits:
    json_list = []
    toon_list = []
    for row in tqdm(dataset[split]):
        ground_truth = json.loads(row['ground_truth'])['gt_parse']
        json_list.append(json.dumps(ground_truth))
        toon_list.append(encode(ground_truth))
    results[split] = {"json_list": json_list, "toon_list": toon_list}


100%|██████████| 100/100 [00:04<00:00, 24.38it/s]


In [9]:
! pip install --user -qU tiktoken

In [19]:
import tiktoken
from tqdm import tqdm

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

token_results = {}
for split in splits:
    json_list = results[split]["json_list"]
    toon_list = results[split]["toon_list"]

    json_tokens = 0
    for json_string in tqdm(json_list):
        json_tokens += num_tokens_from_string(json_string, "cl100k_base")

    toon_tokens = 0
    for toon_string in tqdm(toon_list):
        toon_tokens += num_tokens_from_string(toon_string, "cl100k_base")

    token_results[f'{split}_json_tokens'] = json_tokens
    token_results[f'{split}_toon_tokens'] = toon_tokens


100%|██████████| 100/100 [00:00<00:00, 7408.99it/s]


In [23]:
import plotly.express as px
import pandas as pd

splits = ['train', 'test', 'validation']
json_vals = [token_results[f"{split}_json_tokens"] for split in splits]
toon_vals = [token_results[f"{split}_toon_tokens"] for split in splits]

# Prepare data in long format for Plotly Express
df = pd.DataFrame({
    'split': splits * 2,
    'tokens': json_vals + toon_vals,
    'format': ['json'] * len(splits) + ['toon'] * len(splits)
})

fig = px.bar(
    df,
    x="split",
    y="tokens",
    color="format",
    barmode="group",
    labels={"tokens": "Token Count", "split": "Dataset Split", "format": "Format"},
    title="Token Count: json vs toon per split"
)
fig.update_layout(width=800, height=600)
fig.show()

d:\anaconda3\Lib\site-packages\plotly\express\_core.py:1979: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  sf: grouped.get_group(s if len(s) > 1 else s[0])


In [29]:
json_tokens_sum = df[df['format'] == 'json']['tokens'].sum()
toon_tokens_sum = df[df['format'] == 'toon']['tokens'].sum()
print(f"JSON Tokens Sum: {json_tokens_sum}")
print(f"Toon Tokens Sum: {toon_tokens_sum}")

gpt_5_cost = 10.0

estimated_json_cost = json_tokens_sum * gpt_5_cost / 10 ** 6
estimated_toon_cost = toon_tokens_sum * gpt_5_cost / 10 ** 6

print(f"Estimated JSON Cost: ${estimated_json_cost:.6f}")
print(f"Estimated Toon Cost: ${estimated_toon_cost:.6f}")  


JSON Tokens Sum: 125739
Toon Tokens Sum: 105491
Estimated JSON Cost: $1.257390
Estimated Toon Cost: $1.054910
